In [2]:
%pip -q install duckdb pyarrow

from google.colab import drive
from pathlib import Path
import os

import duckdb
import pyarrow.parquet as pq

drive.mount("/content/drive", force_remount=False)

DATA_DIR = Path("/content/drive/MyDrive/Language Detection")
PARQUET_PATH = DATA_DIR / "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
TEMP_DIR = Path("/content/duckdb_tmp")

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

TEMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET threads = {max(1, min(os.cpu_count() or 4, 8))}")
con.execute("SET memory_limit = '4GB'")
con.execute(f"SET temp_directory = '{TEMP_DIR.as_posix()}'")
con.execute("SET preserve_insertion_order = false")

parquet = pq.ParquetFile(PARQUET_PATH)
metadata = parquet.metadata

print("File :", PARQUET_PATH)
print("Size :", f"{PARQUET_PATH.stat().st_size / 1024**2:.2f} MB")
print("Rows :", f"{metadata.num_rows:,}")
print("Cols :", metadata.num_columns)
print()
print(parquet.schema)

Mounted at /content/drive
File : /content/drive/MyDrive/Language Detection/sessions_lang_transcript_2026-08-23_2026-08-24.parquet
Size : 469.49 MB
Rows : 3,469
Cols : 12

required group field_id=-1 schema {
  optional int64 field_id=-1 gamesession_id;
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 game_name (String);
  optional binary field_id=-1 url (String);
  optional binary field_id=-1 model_type (String);
  optional binary field_id=-1 created_at (String);
  optional binary field_id=-1 lang_detected (String);
  optional double field_id=-1 lang_probability;
  optional group field_id=-1 transcript_segments (List) {
    repeated group field_id=-1 list {
      optional group field_id=-1 element {
        optional binary field_id=-1 text (String);
        optional group field_id=-1 timestamp (List) {
          repeated group field_id=-1 list {
            optional double field_id=-1 element;
          }
        }
        optional group field_id=-1 words (List) {
   

In [3]:
sample = con.execute(
    f"""
    SELECT
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        created_at,
        lang_detected,
        lang_probability,
        len(transcript_segments) AS segment_count
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    ORDER BY
        TRY_CAST(created_at AS TIMESTAMP) DESC NULLS LAST,
        gamesession_id DESC
    LIMIT 20
    """
).df()

with __import__("pandas").option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(sample)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gamesession_id,user_id,game_name,url,model_type,created_at,lang_detected,lang_probability,segment_count
0,141268501,424509,Quarantine Zone: The Last Check,https://www.twitch.tv/videos/2854311727,gen10,2026-08-23 23:59:46,en,0.9863,458
1,141268683,805510,COD: Warzone3-2,https://www.twitch.tv/videos/2854243785,warfare2,2026-08-23 23:59:31,en,0.9961,398
2,141268775,694829,Grand Theft Auto V,https://www.twitch.tv/videos/2854275678,gen10,2026-08-23 23:59:27,en,0.7856,13
3,141267410,185094,Dune: Awakening,https://www.twitch.tv/videos/2854174984,gen10,2026-08-23 23:59:20,en,0.9985,1582
4,141192575,666768,COD: Warzone3-2,https://www.twitch.tv/videos/2852368077,warfare2,2026-08-23 23:59:09,de,0.8784,348
5,141268793,5702,COD: Warzone3-2,https://www.twitch.tv/videos/2854356670,warfare2,2026-08-23 23:58:04,en,0.9683,129
6,141268467,386936,COD: Modern Warfare III,https://www.twitch.tv/videos/2854318859,gen10,2026-08-23 23:57:01,en,0.9990,234
7,141268071,856479,Phasmophobia,https://www.twitch.tv/videos/2853103396,gen10,2026-08-23 23:56:22,en,0.9985,438
8,141268194,838126,Star Citizen,https://www.twitch.tv/videos/2854141333,gen10,2026-08-23 23:56:20,de,0.9961,1327
9,141268553,427268,Fortnite,https://eklipse-streamscope-upload-us-prod.eklipse.gg/upload_streams/beabe75d-6a0a-4361-a3c4-4bf4c5c1c6af.mp4,fortnite,2026-08-23 23:55:57,en,0.6875,149


In [5]:
import pandas as pd

LANGUAGES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

language_sql = ", ".join(f"'{code}'" for code in LANGUAGES)

sessions_by_language = con.execute(
    f"""
    SELECT
        lang_detected AS language_code,
        gamesession_id,
        len(transcript_segments) AS segment_count,
        TRY_CAST(created_at AS TIMESTAMP) AS created_at,
        url
    FROM read_parquet('{PARQUET_PATH.as_posix()}')
    WHERE
        lang_detected IN ({language_sql})
        AND transcript_segments IS NOT NULL
    ORDER BY
        lang_detected,
        TRY_CAST(created_at AS TIMESTAMP) DESC NULLS LAST,
        gamesession_id DESC
    """
).df()

sessions_by_language.insert(
    0,
    "language",
    sessions_by_language["language_code"].map(LANGUAGES),
)

language_summary = (
    sessions_by_language
    .groupby(
        ["language", "language_code"],
        sort=False,
        as_index=False,
    )
    .agg(
        total_sessions=("gamesession_id", "size"),
        total_segments=("segment_count", "sum"),
        min_segments=("segment_count", "min"),
        avg_segments=("segment_count", "mean"),
        max_segments=("segment_count", "max"),
    )
)

language_summary["avg_segments"] = (
    language_summary["avg_segments"].round(2)
)

print("SUMMARY PER LANGUAGE")
display(language_summary)

for code, language in LANGUAGES.items():
    frame = (
        sessions_by_language
        .loc[
            sessions_by_language["language_code"].eq(code),
            [
                "gamesession_id",
                "segment_count",
                "created_at",
                "url",
            ],
        ]
        .reset_index(drop=True)
    )

    print()
    print(f"{language.upper()} ({code}) — {len(frame):,} sessions")

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", None,
    ):
        display(frame)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SUMMARY PER LANGUAGE


,language,language_code,total_sessions,total_segments,min_segments,avg_segments,max_segments
0,German,de,142,134637,0,948.15,4412
1,English,en,2847,2202785,0,773.72,4437
2,Spanish,es,147,94665,0,643.98,2746
3,French,fr,83,55135,2,664.28,1806
4,Portuguese,pt,67,57304,52,855.28,3426
5,Russian,ru,55,39816,0,723.93,2897



ENGLISH (en) — 2,847 sessions


,gamesession_id,segment_count,created_at,url
0,141268501,458,2026-08-23 23:59:46,https://www.twitch.tv/videos/2854311727
1,141268683,398,2026-08-23 23:59:31,https://www.twitch.tv/videos/2854243785
2,141268775,13,2026-08-23 23:59:27,https://www.twitch.tv/videos/2854275678
3,141267410,1582,2026-08-23 23:59:20,https://www.twitch.tv/videos/2854174984
4,141268793,129,2026-08-23 23:58:04,https://www.twitch.tv/videos/2854356670
5,141268467,234,2026-08-23 23:57:01,https://www.twitch.tv/videos/2854318859
6,141268071,438,2026-08-23 23:56:22,https://www.twitch.tv/videos/2853103396
7,141268553,149,2026-08-23 23:55:57,https://eklipse-streamscope-upload-us-prod.eklipse.gg/upload_streams/beabe75d-6a0a-4361-a3c4-4bf4c5c1c6af.mp4
8,141261607,1631,2026-08-23 23:55:53,https://www.youtube.com/watch?v=bvq10rcghiQ
9,141221439,1719,2026-08-23 23:54:01,https://kick.com/jahjah-marquez/videos/e4700dc5-f8e9-4460-bd39-4bf251431593


...

,gamesession_id,segment_count,created_at,url
0,141268410,510,2026-08-23 23:44:11,https://www.twitch.tv/videos/2854267348
1,141266587,938,2026-08-23 23:22:19,https://www.twitch.tv/videos/2854208164
2,141266589,241,2026-08-23 23:09:18,https://www.twitch.tv/videos/2854237020
3,141266163,1488,2026-08-23 22:41:56,https://www.twitch.tv/videos/2853774498
4,141266646,64,2026-08-23 22:19:01,https://www.twitch.tv/videos/2854203171
5,141262388,1089,2026-08-23 22:09:19,https://www.youtube.com/watch?v=MDLhox_0yKk
6,141265858,190,2026-08-23 22:06:07,https://www.twitch.tv/videos/2854201589
7,141261605,951,2026-08-23 21:49:29,https://www.youtube.com/watch?v=DPM1h5wViaw
8,141266504,59,2026-08-23 21:49:00,https://www.twitch.tv/videos/2854243609
9,141264862,668,2026-08-23 19:43:54,https://www.twitch.tv/videos/2854130308



SPANISH (es) — 147 sessions


,gamesession_id,segment_count,created_at,url
0,141268448,948,2026-08-23 23:55:21,https://www.twitch.tv/videos/2853628924
1,141267802,125,2026-08-23 23:14:42,https://www.twitch.tv/videos/2854317733
2,141267465,774,2026-08-23 23:04:01,https://www.twitch.tv/videos/2853842062
3,141267612,180,2026-08-23 23:02:23,https://www.twitch.tv/videos/2854297554
4,141266154,456,2026-08-23 22:54:57,https://www.twitch.tv/videos/2854196517
5,141253792,333,2026-08-23 22:49:29,https://www.twitch.tv/videos/2853731045
6,141261474,1392,2026-08-23 22:44:01,https://www.youtube.com/watch?v=KtAWOITWjqk
7,141266980,1598,2026-08-23 22:43:44,https://www.twitch.tv/videos/2853694264?platformId=4&userId=856416
8,141266763,152,2026-08-23 22:26:31,https://www.twitch.tv/videos/2853758265
9,141266765,443,2026-08-23 22:14:06,https://www.twitch.tv/videos/2853829944



RUSSIAN (ru) — 55 sessions


,gamesession_id,segment_count,created_at,url
0,141267623,1354,2026-08-23 23:32:57,https://www.twitch.tv/videos/2852396591?platformId=4&userId=856454
1,141266950,207,2026-08-23 22:19:45,https://www.twitch.tv/videos/2854259378
2,141265782,474,2026-08-23 22:03:52,https://www.twitch.tv/videos/2854197221
3,141266307,254,2026-08-23 22:01:54,https://www.twitch.tv/videos/2854084381
4,141265209,545,2026-08-23 21:19:55,https://www.twitch.tv/videos/2854154525
5,141264593,890,2026-08-23 20:35:57,https://www.twitch.tv/videos/2854120409
6,141264850,490,2026-08-23 20:17:59,https://www.twitch.tv/videos/2854112578
7,141264221,643,2026-08-23 19:48:21,https://www.twitch.tv/videos/2854084941
8,141263750,743,2026-08-23 19:08:17,https://www.twitch.tv/videos/2854087744
9,141262376,85,2026-08-23 18:40:44,https://www.twitch.tv/videos/2853330048


In [6]:
MIN_SEGMENTS = 100

for code, language in LANGUAGES.items():
    frame = (
        sessions_by_language
        .loc[
            (sessions_by_language["language_code"] == code)
            & (sessions_by_language["segment_count"] >= MIN_SEGMENTS),
            [
                "gamesession_id",
                "segment_count",
                "created_at",
                "url",
            ],
        ]
        .sort_values(
            ["segment_count", "created_at"],
            ascending=[True, False],
        )
        .reset_index(drop=True)
    )

    print()
    print(f"{language.upper()} ({code}) — {len(frame):,} sessions — minimum {MIN_SEGMENTS} segments")
    display(frame)


ENGLISH (en) — 2,607 sessions — minimum 100 segments


,gamesession_id,segment_count,created_at,url
0,141157903,100,2026-08-23 19:21:18,https://www.twitch.tv/videos/2849979256
1,141253050,100,2026-08-23 11:00:03,https://kick.com/littguy4life/videos/82686667-...
2,141191499,100,2026-08-23 09:48:52,https://www.twitch.tv/videos/2852325375
3,141247735,100,2026-08-23 09:31:42,https://www.twitch.tv/videos/2851717970
4,141249269,100,2026-08-23 09:20:07,https://www.twitch.tv/videos/2853876575
...,...,...,...,...
2602,141242817,4224,2026-08-23 06:59:30,https://www.twitch.tv/videos/2851822475?platfo...
2603,141254293,4233,2026-08-23 12:35:01,https://www.twitch.tv/videos/2853551419
2604,141238222,4277,2026-08-23 09:48:23,https://www.youtube.com/watch?v=hF82Qp4KqQY
2605,141260625,4384,2026-08-23 16:29:28,https://kick.com/sgt_jackson12/videos/050a9dc7...



GERMAN (de) — 135 sessions — minimum 100 segments


,gamesession_id,segment_count,created_at,url
0,141264210,125,2026-08-23 18:39:26,https://www.twitch.tv/videos/2854157061
1,141268064,126,2026-08-23 23:29:07,https://www.twitch.tv/videos/2854317619
2,141238626,131,2026-08-23 04:09:16,https://www.twitch.tv/videos/2853620823
3,141265651,147,2026-08-23 21:53:10,https://www.twitch.tv/videos/2854208944
4,141236998,182,2026-08-23 03:22:23,https://www.twitch.tv/videos/2853489925
...,...,...,...,...
130,141246213,3266,2026-08-23 08:37:26,https://kick.com/ilvyauri/videos/2a7c20fd-ea7d...
131,141252746,3267,2026-08-23 11:09:38,https://www.twitch.tv/videos/2853446005
132,141238377,3514,2026-08-23 11:00:27,https://www.youtube.com/watch?v=92egO0NFDX4
133,141234763,3729,2026-08-23 02:50:50,https://www.twitch.tv/videos/2853123772



FRENCH (fr) — 77 sessions — minimum 100 segments


...

,gamesession_id,segment_count,created_at,url
0,141251341,122,2026-08-23 11:08:07,https://www.twitch.tv/videos/2853835341
1,141252406,130,2026-08-23 10:33:33,https://www.twitch.tv/videos/2853868375
2,141252827,132,2026-08-23 10:45:39,https://www.twitch.tv/videos/2853909460
3,141250441,142,2026-08-23 09:35:24,https://www.twitch.tv/videos/2853841196
4,141265858,190,2026-08-23 22:06:07,https://www.twitch.tv/videos/2854201589
...,...,...,...,...
58,141246627,2323,2026-08-23 09:46:56,https://www.twitch.tv/videos/2853536190
59,141249378,2368,2026-08-23 11:09:09,https://kick.com/kakaiju/videos/145cdba9-d985-...
60,141254764,2377,2026-08-23 13:51:24,https://www.twitch.tv/videos/2853718927
61,141259491,2688,2026-08-23 15:49:43,https://www.twitch.tv/videos/2853675248



SPANISH (es) — 138 sessions — minimum 100 segments


,gamesession_id,segment_count,created_at,url
0,141247929,100,2026-08-23 08:36:10,https://www.twitch.tv/videos/2853837750
1,141233178,105,2026-08-23 06:05:19,https://www.youtube.com/watch?v=Zl3Qc-RBau0
2,141256940,118,2026-08-23 12:39:53,https://www.twitch.tv/videos/2853996303
3,141231518,118,2026-08-23 05:49:19,https://www.youtube.com/watch?v=z9EBy2BfHkw
4,141248993,120,2026-08-23 09:01:50,https://eklipse-streamscope-upload-us-prod.ekl...
...,...,...,...,...
133,141242594,1729,2026-08-23 06:11:12,https://www.twitch.tv/videos/2853337833
134,141231704,1753,2026-08-23 03:42:50,https://kick.com/yunalunatica/videos/b53cbc8f-...
135,141237677,1921,2026-08-23 03:48:28,https://www.twitch.tv/videos/2853336564
136,141225226,2729,2026-08-23 00:19:05,https://www.youtube.com/watch?v=PhbierZc1lQ



RUSSIAN (ru) — 53 sessions — minimum 100 segments


,gamesession_id,segment_count,created_at,url
0,141228447,117,2026-08-23 00:28:08,https://eklipse-streamscope-upload-us-prod.ekl...
1,141260682,120,2026-08-23 14:42:10,https://www.twitch.tv/videos/2852572934
2,141235885,158,2026-08-23 03:43:07,https://www.twitch.tv/videos/2853460554
3,141262832,199,2026-08-23 16:41:49,https://eklipse-streamscope-upload-us-prod.ekl...
4,141250010,204,2026-08-23 09:25:24,https://www.twitch.tv/videos/2853621396
5,141266950,207,2026-08-23 22:19:45,https://www.twitch.tv/videos/2854259378
6,141238692,242,2026-08-23 04:05:38,https://www.twitch.tv/videos/2849746374?platfo...
7,141236825,251,2026-08-23 04:34:11,https://www.twitch.tv/videos/2853442316
8,141266307,254,2026-08-23 22:01:54,https://www.twitch.tv/videos/2854084381
9,141240767,288,2026-08-23 05:15:18,https://www.twitch.tv/videos/2853564486


In [2]:
from pathlib import Path
import os

import duckdb
import pandas as pd

PARQUET_PATH = Path(
    "/content/drive/MyDrive/Language Detection/"
    "sessions_lang_transcript_2026-08-23_2026-08-24.parquet"
)

LANGUAGES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

MIN_WORDS_PER_SEGMENT = 4
MIN_SEGMENTS_PER_SESSION = 100

if not PARQUET_PATH.is_file():
    raise FileNotFoundError(PARQUET_PATH)

con = duckdb.connect()

con.execute(f"SET threads = {max(1, min(os.cpu_count() or 2, 4))}")
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")

language_sql = ", ".join(f"'{code}'" for code in LANGUAGES)

filtered_sessions = con.execute(
    f"""
    WITH session_counts AS (
        SELECT
            lang_detected AS language_code,
            gamesession_id,
            url,
            TRY_CAST(created_at AS TIMESTAMP) AS created_at,
            list_count(
                list_filter(
                    transcript_segments,
                    segment ->
                        segment.words IS NOT NULL
                        AND len(segment.words) >= {MIN_WORDS_PER_SEGMENT}
                )
            ) AS segment_count
        FROM read_parquet(
            '{PARQUET_PATH.as_posix()}',
            union_by_name = false
        )
        WHERE
            lang_detected IN ({language_sql})
            AND transcript_segments IS NOT NULL
            AND len(transcript_segments) >= {MIN_SEGMENTS_PER_SESSION}
    )
    SELECT
        language_code,
        gamesession_id,
        url,
        created_at,
        segment_count
    FROM session_counts
    WHERE segment_count >= {MIN_SEGMENTS_PER_SESSION}
    ORDER BY
        language_code ASC,
        segment_count ASC,
        created_at DESC NULLS LAST,
        gamesession_id DESC
    """
).df()

filtered_sessions.insert(
    0,
    "language",
    filtered_sessions["language_code"].map(LANGUAGES),
)

for code, language in LANGUAGES.items():
    frame = (
        filtered_sessions
        .loc[
            filtered_sessions["language_code"].eq(code),
            [
                "gamesession_id",
                "segment_count",
                "created_at",
                "url",
            ],
        ]
        .reset_index(drop=True)
    )

    print()
    print(f"{language.upper()} ({code}) — {len(frame):,} sessions")

    display(frame)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


ENGLISH (en) — 2,510 sessions


,gamesession_id,segment_count,created_at,url
0,141245536,100,2026-08-23 08:14:23,https://kick.com/boolinofftheperky/videos/9d67...
1,141242971,100,2026-08-23 06:07:43,https://eklipse-streamscope-upload-us-prod.ekl...
2,141264751,101,2026-08-23 19:33:54,https://www.twitch.tv/videos/2854142885
3,141258632,101,2026-08-23 19:06:49,https://www.youtube.com/watch?v=hjqISZ4EobY
4,141253180,101,2026-08-23 12:11:04,https://www.twitch.tv/videos/2848541982
...,...,...,...,...
2505,141242817,3990,2026-08-23 06:59:30,https://www.twitch.tv/videos/2851822475?platfo...
2506,141238222,4001,2026-08-23 09:48:23,https://www.youtube.com/watch?v=hF82Qp4KqQY
2507,141260625,4004,2026-08-23 16:29:28,https://kick.com/sgt_jackson12/videos/050a9dc7...
2508,141254293,4043,2026-08-23 12:35:01,https://www.twitch.tv/videos/2853551419



GERMAN (de) — 134 sessions


,gamesession_id,segment_count,created_at,url
0,141264210,107,2026-08-23 18:39:26,https://www.twitch.tv/videos/2854157061
1,141265651,114,2026-08-23 21:53:10,https://www.twitch.tv/videos/2854208944
2,141236998,115,2026-08-23 03:22:23,https://www.twitch.tv/videos/2853489925
3,141238626,118,2026-08-23 04:09:16,https://www.twitch.tv/videos/2853620823
4,141236793,120,2026-08-23 03:08:02,https://www.twitch.tv/n00bkrieger/v/2853489925...
...,...,...,...,...
129,141252746,2931,2026-08-23 11:09:38,https://www.twitch.tv/videos/2853446005
130,141246213,3069,2026-08-23 08:37:26,https://kick.com/ilvyauri/videos/2a7c20fd-ea7d...
131,141238377,3423,2026-08-23 11:00:27,https://www.youtube.com/watch?v=92egO0NFDX4
132,141234763,3477,2026-08-23 02:50:50,https://www.twitch.tv/videos/2853123772


...

,gamesession_id,segment_count,created_at,url
0,141266589,159,2026-08-23 23:09:18,https://www.twitch.tv/videos/2854237020
1,141265858,164,2026-08-23 22:06:07,https://www.twitch.tv/videos/2854201589
2,141174273,174,2026-08-23 09:23:31,https://www.youtube.com/watch?v=6-Pzf840Auo
3,141247850,192,2026-08-23 09:47:14,https://www.twitch.tv/videos/2853754766
4,141247930,213,2026-08-23 08:39:47,https://eklipse-streamscope-upload-us-prod.s3....
5,141250021,230,2026-08-23 10:11:32,https://www.twitch.tv/gabepeixe/v/2853777186?p...
6,141241196,249,2026-08-23 05:51:20,https://eklipse-streamscope-upload-us-prod.ekl...
7,141239739,250,2026-08-23 04:52:26,https://eklipse-streamscope-upload-us-prod.ekl...
8,141204400,254,2026-08-23 05:43:50,https://www.twitch.tv/videos/2852627350
9,141243870,340,2026-08-23 08:19:43,https://www.twitch.tv/videos/2853700659



SPANISH (es) — 134 sessions


,gamesession_id,segment_count,created_at,url
0,141242562,100,2026-08-23 07:04:11,https://www.twitch.tv/videos/2853679816
1,141244879,106,2026-08-23 07:11:38,https://www.twitch.tv/videos/2853750121
2,141248993,107,2026-08-23 09:01:50,https://eklipse-streamscope-upload-us-prod.ekl...
3,141231518,108,2026-08-23 05:49:19,https://www.youtube.com/watch?v=z9EBy2BfHkw
4,141267802,111,2026-08-23 23:14:42,https://www.twitch.tv/videos/2854317733
...,...,...,...,...
129,141266980,1545,2026-08-23 22:43:44,https://www.twitch.tv/videos/2853694264?platfo...
130,141231704,1640,2026-08-23 03:42:50,https://kick.com/yunalunatica/videos/b53cbc8f-...
131,141237677,1740,2026-08-23 03:48:28,https://www.twitch.tv/videos/2853336564
132,141225226,2449,2026-08-23 00:19:05,https://www.youtube.com/watch?v=PhbierZc1lQ



RUSSIAN (ru) — 51 sessions


,gamesession_id,segment_count,created_at,url
0,141235885,120,2026-08-23 03:43:07,https://www.twitch.tv/videos/2853460554
1,141266950,157,2026-08-23 22:19:45,https://www.twitch.tv/videos/2854259378
2,141262832,157,2026-08-23 16:41:49,https://eklipse-streamscope-upload-us-prod.ekl...
3,141236825,195,2026-08-23 04:34:11,https://www.twitch.tv/videos/2853442316
4,141250010,201,2026-08-23 09:25:24,https://www.twitch.tv/videos/2853621396
5,141266307,207,2026-08-23 22:01:54,https://www.twitch.tv/videos/2854084381
6,141240767,220,2026-08-23 05:15:18,https://www.twitch.tv/videos/2853564486
7,141238692,229,2026-08-23 04:05:38,https://www.twitch.tv/videos/2849746374?platfo...
8,141231053,237,2026-08-23 00:10:52,https://www.twitch.tv/videos/2853287483
9,141235466,271,2026-08-23 03:38:02,https://www.twitch.tv/videos/2853505768


In [4]:
LANGUAGES = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt": "Portuguese",
    "es": "Spanish",
    "ru": "Russian",
}

MIN_WORDS = 4
MIN_SEGMENTS = 100
SESSIONS_PER_LANGUAGE = 5

language_sql = ", ".join(f"'{code}'" for code in LANGUAGES)

selected_sessions = con.execute(
    f"""
    WITH session_counts AS (
        SELECT
            gamesession_id,
            user_id,
            game_name,
            url,
            model_type,
            TRY_CAST(created_at AS TIMESTAMP) AS created_at,
            lang_detected AS language_code,
            lang_probability,
            list_count(
                list_filter(
                    transcript_segments,
                    segment ->
                        segment.words IS NOT NULL
                        AND len(segment.words) >= {MIN_WORDS}
                )
            ) AS segment_count
        FROM read_parquet('{PARQUET_PATH.as_posix()}')
        WHERE
            lang_detected IN ({language_sql})
            AND transcript_segments IS NOT NULL
            AND len(transcript_segments) >= {MIN_SEGMENTS}
    ),
    eligible AS (
        SELECT *
        FROM session_counts
        WHERE segment_count >= {MIN_SEGMENTS}
    ),
    ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY language_code
                ORDER BY
                    segment_count ASC,
                    created_at DESC NULLS LAST,
                    gamesession_id DESC
            ) AS session_rank
        FROM eligible
    )
    SELECT
        language_code,
        session_rank,
        gamesession_id,
        user_id,
        game_name,
        url,
        model_type,
        created_at,
        lang_probability,
        segment_count
    FROM ranked
    WHERE session_rank <= {SESSIONS_PER_LANGUAGE}
    ORDER BY
        language_code,
        session_rank
    """
).df()

selected_sessions.insert(
    0,
    "language",
    selected_sessions["language_code"].map(LANGUAGES),
)

for code, language in LANGUAGES.items():
    frame = (
        selected_sessions[
            selected_sessions["language_code"].eq(code)
        ]
        .drop(columns=["language", "language_code"])
        .reset_index(drop=True)
    )

    print()
    print(f"{language.upper()} ({code}) — {len(frame)} sessions")

    display(frame)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


ENGLISH (en) — 5 sessions


,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,1,141245536,639030,Just Chatting,https://kick.com/boolinofftheperky/videos/9d67...,gen10,2026-08-23 08:14:23,0.9038,100
1,2,141242971,855243,COD: Warzone3-2,https://eklipse-streamscope-upload-us-prod.ekl...,warfare2,2026-08-23 06:07:43,0.9106,100
2,3,141264751,788072,Call of Duty: Modern Warfare 4,https://www.twitch.tv/videos/2854142885,warfare4,2026-08-23 19:33:54,0.8794,101
3,4,141258632,81536,Other Games,https://www.youtube.com/watch?v=hjqISZ4EobY,gen10,2026-08-23 19:06:49,0.9971,101
4,5,141253180,853240,Gears 5,https://www.twitch.tv/videos/2848541982,gen5,2026-08-23 12:11:04,0.9258,101



GERMAN (de) — 5 sessions


,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,1,141264210,399263,Call of Duty: Modern Warfare 4,https://www.twitch.tv/videos/2854157061,warfare4,2026-08-23 18:39:26,0.9072,107
1,2,141265651,687197,Escape from Tarkov,https://www.twitch.tv/videos/2854208944,gen10,2026-08-23 21:53:10,0.9199,114
2,3,141236998,833542,COD: Warzone3-2,https://www.twitch.tv/videos/2853489925,warfare2,2026-08-23 03:22:23,0.6826,115
3,4,141238626,812794,COD: Warzone3-2,https://www.twitch.tv/videos/2853620823,warfare2,2026-08-23 04:09:16,0.9678,118
4,5,141236793,833542,COD: Warzone3-2,https://www.twitch.tv/n00bkrieger/v/2853489925...,warfare2,2026-08-23 03:08:02,0.6826,120


...

,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,1,141266589,853110,Other Games,https://www.twitch.tv/videos/2854237020,gen5,2026-08-23 23:09:18,0.9951,159
1,2,141265858,853248,Arena Breakout: Infinite,https://www.twitch.tv/videos/2854201589,gen5,2026-08-23 22:06:07,0.9834,164
2,3,141174273,588093,Mortal Shell 2,https://www.youtube.com/watch?v=6-Pzf840Auo,gen9,2026-08-23 09:23:31,0.8574,174
3,4,141247850,853884,SnowRunner,https://www.twitch.tv/videos/2853754766,gen5,2026-08-23 09:47:14,0.9980,192
4,5,141247930,837349,Dead by Daylight,https://eklipse-streamscope-upload-us-prod.s3....,gen10,2026-08-23 08:39:47,0.9951,213



SPANISH (es) — 5 sessions


,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,1,141242562,854359,League of Legends: Wild Rift,https://www.twitch.tv/videos/2853679816,gen5,2026-08-23 07:04:11,0.9321,100
1,2,141244879,821148,COD: Warzone3-2,https://www.twitch.tv/videos/2853750121,warfare2,2026-08-23 07:11:38,0.9136,106
2,3,141248993,855905,EA Sports FC 26 & 27,https://eklipse-streamscope-upload-us-prod.ekl...,gen4,2026-08-23 09:01:50,0.9961,107
3,4,141231518,841253,Fortnite,https://www.youtube.com/watch?v=z9EBy2BfHkw,fortnite,2026-08-23 05:49:19,0.9966,108
4,5,141267802,847747,EA Sports UFC 6,https://www.twitch.tv/videos/2854317733,gen10,2026-08-23 23:14:42,0.9312,111



RUSSIAN (ru) — 5 sessions


,session_rank,gamesession_id,user_id,game_name,url,model_type,created_at,lang_probability,segment_count
0,1,141235885,851755,DayZ,https://www.twitch.tv/videos/2853460554,gen5,2026-08-23 03:43:07,0.9663,120
1,2,141266950,617806,Apex Legends,https://www.twitch.tv/videos/2854259378,apex,2026-08-23 22:19:45,0.9805,157
2,3,141262832,856164,Dota 2,https://eklipse-streamscope-upload-us-prod.ekl...,gen4,2026-08-23 16:41:49,0.9956,157
3,4,141236825,853661,Valorant,https://www.twitch.tv/videos/2853442316,gen5,2026-08-23 04:34:11,0.9531,195
4,5,141250010,855938,Just Chatting,https://www.twitch.tv/videos/2853621396,gen5,2026-08-23 09:25:24,0.9961,201
